<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=314362363" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import os
from collections import Counter, defaultdict
from itertools import product

# ============================================================================
# 0. SETUP
# ============================================================================
is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(path) as f:
    data = json.load(f)

# ============================================================================
# 1. GRID UTILITIES
# ============================================================================
def copy_grid(g):
    return [row[:] for row in g]

def grid_dims(g):
    return (len(g), len(g[0])) if g and g[0] else (0, 0)

def flatten(g):
    return [c for row in g for c in row]

def grids_equal(a, b):
    if len(a) != len(b):
        return False
    return all(ra == rb for ra, rb in zip(a, b))

def unique_colors(g):
    return set(flatten(g))

def most_common_color(grid):
    return Counter(flatten(grid)).most_common(1)[0][0]

def background_color(grid):
    return most_common_color(grid)

def color_counts(grid):
    return Counter(flatten(grid))

def grid_to_tuple(g):
    return tuple(tuple(r) for r in g)

def make_grid(h, w, color=0):
    return [[color] * w for _ in range(h)]

def set_cell(grid, r, c, color):
    if 0 <= r < len(grid) and 0 <= c < len(grid[0]):
        grid[r][c] = color

def get_cell(grid, r, c, default=0):
    if 0 <= r < len(grid) and 0 <= c < len(grid[0]):
        return grid[r][c]
    return default

def color_positions(grid, color):
    return [(i, j) for i in range(len(grid))
            for j in range(len(grid[0])) if grid[i][j] == color]

def non_bg_colors(grid):
    bg = background_color(grid)
    return unique_colors(grid) - {bg}

# ============================================================================
# 2. GEOMETRIC TRANSFORMS
# ============================================================================
def rotate90(g):
    return list(map(list, zip(*g[::-1])))

def rotate180(g):
    return rotate90(rotate90(g))

def rotate270(g):
    return rotate90(rotate90(rotate90(g)))

def flip_h(g):
    return [row[::-1] for row in g]

def flip_v(g):
    return g[::-1]

def transpose(g):
    h, w = grid_dims(g)
    return [[g[i][j] for i in range(h)] for j in range(w)]

def transpose_anti(g):
    return rotate90(flip_v(g))

ALL_RIGID_TRANSFORMS = [
    ("identity",       lambda x: copy_grid(x)),
    ("rot90",          rotate90),
    ("rot180",         rotate180),
    ("rot270",         rotate270),
    ("flip_h",         flip_h),
    ("flip_v",         flip_v),
    ("transpose",      transpose),
    ("transpose_anti", transpose_anti),
]

# ============================================================================
# 3. COLOR MAPPING
# ============================================================================
def color_map(g, mapping):
    return [[mapping.get(c, c) for c in row] for row in g]

def infer_color_map(train):
    mapping = {}
    for pair in train:
        inp, out = pair["input"], pair["output"]
        h = min(len(inp), len(out))
        w = min(len(inp[0]), len(out[0]))
        for i in range(h):
            for j in range(w):
                mapping[inp[i][j]] = out[i][j]
    return mapping

def infer_color_map_robust(train):
    votes = defaultdict(list)
    for pair in train:
        inp, out = pair["input"], pair["output"]
        h = min(len(inp), len(out))
        w = min(len(inp[0]), len(out[0]))
        for i in range(h):
            for j in range(w):
                votes[inp[i][j]].append(out[i][j])
    return {k: Counter(v).most_common(1)[0][0] for k, v in votes.items()}

# ============================================================================
# 4. CROPPING & BOUNDING BOX
# ============================================================================
def bounding_box(grid, ignore_color=None):
    h, w = grid_dims(grid)
    if ignore_color is None:
        ignore_color = background_color(grid)
    min_r, max_r, min_c, max_c = h, -1, w, -1
    for i in range(h):
        for j in range(w):
            if grid[i][j] != ignore_color:
                min_r = min(min_r, i)
                max_r = max(max_r, i)
                min_c = min(min_c, j)
                max_c = max(max_c, j)
    if max_r == -1:
        return 0, 0, h, w
    return min_r, min_c, max_r + 1, max_c + 1

def crop(grid, r1, c1, r2, c2):
    return [row[c1:c2] for row in grid[r1:r2]]

def crop_to_content(grid, ignore_color=None):
    r1, c1, r2, c2 = bounding_box(grid, ignore_color)
    return crop(grid, r1, c1, r2, c2)

def crop_color(grid, color):
    h, w = grid_dims(grid)
    min_r, max_r, min_c, max_c = h, -1, w, -1
    for i in range(h):
        for j in range(w):
            if grid[i][j] == color:
                min_r = min(min_r, i)
                max_r = max(max_r, i)
                min_c = min(min_c, j)
                max_c = max(max_c, j)
    if max_r == -1:
        return [[]]
    return crop(grid, min_r, min_c, max_r + 1, max_c + 1)

# ============================================================================
# 5. TILING & SCALING
# ============================================================================
def tile_grid(g, reps_h, reps_w):
    result = []
    for _ in range(reps_h):
        for row in g:
            new_row = []
            for _ in range(reps_w):
                new_row.extend(row)
            result.append(new_row)
    return result

def scale_grid(g, factor_h, factor_w):
    result = []
    for row in g:
        new_row = []
        for c in row:
            new_row.extend([c] * factor_w)
        for _ in range(factor_h):
            result.append(new_row[:])
    return result

def downscale_grid(g, factor_h, factor_w):
    h, w = grid_dims(g)
    nh, nw = h // factor_h, w // factor_w
    result = []
    for i in range(nh):
        row = []
        for j in range(nw):
            block = [g[i * factor_h + di][j * factor_w + dj]
                     for di in range(factor_h) for dj in range(factor_w)]
            row.append(Counter(block).most_common(1)[0][0])
        result.append(row)
    return result

# ============================================================================
# 6. FLOOD FILL & CONNECTED COMPONENTS
# ============================================================================
def flood_fill_positions(grid, start_r, start_c, color=None, visited=None):
    h, w = grid_dims(grid)
    if color is None:
        color = grid[start_r][start_c]
    if visited is None:
        visited = set()
    stack = [(start_r, start_c)]
    positions = []
    while stack:
        r, c = stack.pop()
        if (r, c) in visited or r < 0 or r >= h or c < 0 or c >= w:
            continue
        if grid[r][c] != color:
            continue
        visited.add((r, c))
        positions.append((r, c))
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            stack.append((r + dr, c + dc))
    return positions

def flood_fill_8way(grid, start_r, start_c, color=None, visited=None):
    h, w = grid_dims(grid)
    if color is None:
        color = grid[start_r][start_c]
    if visited is None:
        visited = set()
    stack = [(start_r, start_c)]
    positions = []
    dirs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    while stack:
        r, c = stack.pop()
        if (r, c) in visited or r < 0 or r >= h or c < 0 or c >= w:
            continue
        if grid[r][c] != color:
            continue
        visited.add((r, c))
        positions.append((r, c))
        for dr, dc in dirs:
            stack.append((r + dr, c + dc))
    return positions

def connected_components(grid, ignore_color=None):
    h, w = grid_dims(grid)
    visited = set()
    components = []
    for i in range(h):
        for j in range(w):
            if (i, j) not in visited and (ignore_color is None or grid[i][j] != ignore_color):
                cc = flood_fill_positions(grid, i, j, visited=visited)
                components.append((grid[i][j], cc))
    return components

def extract_objects(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    comps = connected_components(grid, ignore_color=bg)
    objects = []
    for color, positions in comps:
        if not positions:
            continue
        min_r = min(r for r, c in positions)
        max_r = max(r for r, c in positions)
        min_c = min(c for r, c in positions)
        max_c = max(c for r, c in positions)
        oh = max_r - min_r + 1
        ow = max_c - min_c + 1
        obj_grid = [[bg] * ow for _ in range(oh)]
        for r, c in positions:
            obj_grid[r - min_r][c - min_c] = color
        objects.append({
            "color": color,
            "positions": positions,
            "grid": obj_grid,
            "bbox": (min_r, min_c, max_r + 1, max_c + 1),
            "size": len(positions)
        })
    return objects

def extract_multicolor_objects(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    visited = set()
    objects = []
    for i in range(h):
        for j in range(w):
            if (i, j) not in visited and grid[i][j] != bg:
                stack = [(i, j)]
                positions = []
                while stack:
                    r, c = stack.pop()
                    if (r, c) in visited or r < 0 or r >= h or c < 0 or c >= w:
                        continue
                    if grid[r][c] == bg:
                        continue
                    visited.add((r, c))
                    positions.append((r, c))
                    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        stack.append((r + dr, c + dc))
                if positions:
                    min_r = min(r for r, c in positions)
                    max_r = max(r for r, c in positions)
                    min_c = min(c for r, c in positions)
                    max_c = max(c for r, c in positions)
                    oh = max_r - min_r + 1
                    ow = max_c - min_c + 1
                    obj_grid = [[bg] * ow for _ in range(oh)]
                    for r, c in positions:
                        obj_grid[r - min_r][c - min_c] = grid[r][c]
                    objects.append({
                        "positions": positions,
                        "grid": obj_grid,
                        "bbox": (min_r, min_c, max_r + 1, max_c + 1),
                        "size": len(positions),
                        "colors": set(grid[r][c] for r, c in positions),
                        "color": grid[i][j]
                    })
    return objects

# ============================================================================
# 7. SYMMETRY
# ============================================================================
def make_horizontally_symmetric(g):
    result = copy_grid(g)
    h, w = grid_dims(g)
    bg = background_color(g)
    for i in range(h):
        for j in range(w // 2):
            j2 = w - 1 - j
            if result[i][j] != bg and result[i][j2] == bg:
                result[i][j2] = result[i][j]
            elif result[i][j2] != bg and result[i][j] == bg:
                result[i][j] = result[i][j2]
    return result

def make_vertically_symmetric(g):
    result = copy_grid(g)
    h, w = grid_dims(g)
    bg = background_color(g)
    for i in range(h // 2):
        i2 = h - 1 - i
        for j in range(w):
            if result[i][j] != bg and result[i2][j] == bg:
                result[i2][j] = result[i][j]
            elif result[i2][j] != bg and result[i][j] == bg:
                result[i][j] = result[i2][j]
    return result

def make_4way_symmetric(g):
    return make_vertically_symmetric(make_horizontally_symmetric(g))

def complete_symmetry_bg_aware(grid):
    bg = background_color(grid)
    h, w = grid_dims(grid)
    result = copy_grid(grid)
    for i in range(h):
        for j in range(w):
            j2 = w - 1 - j
            if result[i][j] != bg and result[i][j2] == bg:
                result[i][j2] = result[i][j]
    for i in range(h):
        i2 = h - 1 - i
        for j in range(w):
            if result[i][j] != bg and result[i2][j] == bg:
                result[i2][j] = result[i][j]
    return result

# ============================================================================
# 8. BORDER OPERATIONS
# ============================================================================
def extract_border(grid):
    h, w = grid_dims(grid)
    border = []
    for j in range(w):
        border.append(grid[0][j])
        if h > 1:
            border.append(grid[h - 1][j])
    for i in range(1, h - 1):
        border.append(grid[i][0])
        if w > 1:
            border.append(grid[i][w - 1])
    return border

def remove_border(grid):
    if len(grid) <= 2 or len(grid[0]) <= 2:
        return copy_grid(grid)
    return [row[1:-1] for row in grid[1:-1]]

def add_border(grid, color):
    h, w = grid_dims(grid)
    new_w = w + 2
    result = [[color] * new_w]
    for row in grid:
        result.append([color] + row[:] + [color])
    result.append([color] * new_w)
    return result

# ============================================================================
# 9. GRAVITY
# ============================================================================
def gravity_down(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for j in range(w):
        col_vals = [grid[i][j] for i in range(h) if grid[i][j] != bg]
        for idx, val in enumerate(reversed(col_vals)):
            result[h - 1 - idx][j] = val
    return result

def gravity_up(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for j in range(w):
        col_vals = [grid[i][j] for i in range(h) if grid[i][j] != bg]
        for idx, val in enumerate(col_vals):
            result[idx][j] = val
    return result

def gravity_left(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        row_vals = [grid[i][j] for j in range(w) if grid[i][j] != bg]
        for idx, val in enumerate(row_vals):
            result[i][idx] = val
    return result

def gravity_right(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        row_vals = [grid[i][j] for j in range(w) if grid[i][j] != bg]
        for idx, val in enumerate(reversed(row_vals)):
            result[i][w - 1 - idx] = val
    return result

# ============================================================================
# 10. GRID BOOLEAN OPERATIONS
# ============================================================================
def or_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] != bg:
                result[i][j] = g1[i][j]
            elif g2[i][j] != bg:
                result[i][j] = g2[i][j]
    return result

def and_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] == g2[i][j]:
                result[i][j] = g1[i][j]
    return result

def xor_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] != g2[i][j]:
                result[i][j] = g2[i][j]
    return result

def subtract_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] != bg and g2[i][j] == bg:
                result[i][j] = g1[i][j]
    return result

# ============================================================================
# 11. GRID SPLITTING & CONCAT
# ============================================================================
def split_grid_horizontal(g, n):
    h, _ = grid_dims(g)
    ph = h // n
    return [[row[:] for row in g[i * ph:(i + 1) * ph]] for i in range(n)]

def split_grid_vertical(g, n):
    _, w = grid_dims(g)
    pw = w // n
    return [[row[i * pw:(i + 1) * pw] for row in g] for i in range(n)]

def concat_horizontal(grids):
    if not grids:
        return [[]]
    h = len(grids[0])
    result = []
    for i in range(h):
        row = []
        for g in grids:
            if i < len(g):
                row.extend(g[i])
        result.append(row)
    return result

def concat_vertical(grids):
    result = []
    for g in grids:
        result.extend([row[:] for row in g])
    return result

def split_into_quadrants(g):
    h, w = grid_dims(g)
    mh, mw = h // 2, w // 2
    return [
        crop(g, 0, 0, mh, mw),
        crop(g, 0, mw, mh, w),
        crop(g, mh, 0, h, mw),
        crop(g, mh, mw, h, w),
    ]

# ============================================================================
# 12. REPEATING PATTERN & DIVIDERS
# ============================================================================
def find_repeating_unit(grid):
    h, w = grid_dims(grid)
    for uh in range(1, h + 1):
        if h % uh != 0:
            continue
        for uw in range(1, w + 1):
            if w % uw != 0:
                continue
            unit = [row[:uw] for row in grid[:uh]]
            if grids_equal(tile_grid(unit, h // uh, w // uw), grid):
                if uh < h or uw < w:
                    return unit
    return grid

def detect_grid_divisions(grid):
    h, w = grid_dims(grid)
    h_dividers = [i for i in range(h) if len(set(grid[i])) == 1]
    v_dividers = [j for j in range(w) if len(set(grid[i][j] for i in range(h))) == 1]
    return h_dividers, v_dividers

def split_by_dividers(grid, h_dividers, v_dividers):
    h, w = grid_dims(grid)
    def make_ranges(dividers, total):
        ranges = []
        prev = 0
        for d in sorted(dividers):
            if d > prev:
                ranges.append((prev, d))
            prev = d + 1
        if prev < total:
            ranges.append((prev, total))
        return ranges
    h_ranges = make_ranges(h_dividers, h)
    v_ranges = make_ranges(v_dividers, w)
    return [[crop(grid, r1, c1, r2, c2) for c1, c2 in v_ranges] for r1, r2 in h_ranges]

# ============================================================================
# 13. FILL OPERATIONS
# ============================================================================
def fill_enclosed_regions(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = copy_grid(grid)
    border_reachable = set()
    stack = []
    for i in range(h):
        if grid[i][0] == bg:
            stack.append((i, 0))
        if w > 1 and grid[i][w - 1] == bg:
            stack.append((i, w - 1))
    for j in range(w):
        if grid[0][j] == bg:
            stack.append((0, j))
        if h > 1 and grid[h - 1][j] == bg:
            stack.append((h - 1, j))
    while stack:
        r, c = stack.pop()
        if (r, c) in border_reachable:
            continue
        if r < 0 or r >= h or c < 0 or c >= w:
            continue
        if grid[r][c] != bg:
            continue
        border_reachable.add((r, c))
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            stack.append((r + dr, c + dc))
    for i in range(h):
        for j in range(w):
            if grid[i][j] == bg and (i, j) not in border_reachable:
                neighbors = [
                    grid[i + di][j + dj]
                    for di in range(-2, 3) for dj in range(-2, 3)
                    if 0 <= i + di < h and 0 <= j + dj < w and grid[i + di][j + dj] != bg
                ]
                if neighbors:
                    result[i][j] = Counter(neighbors).most_common(1)[0][0]
    return result

def replace_color(grid, old_color, new_color):
    return [[new_color if c == old_color else c for c in row] for row in grid]

def shift_grid(grid, dr, dc, bg=0):
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            ni, nj = i + dr, j + dc
            if 0 <= ni < h and 0 <= nj < w:
                result[ni][nj] = grid[i][j]
    return result

# ============================================================================
# 14. ROW/COL OPERATIONS
# ============================================================================
def duplicate_rows(grid, n):
    result = []
    for row in grid:
        for _ in range(n):
            result.append(row[:])
    return result

def duplicate_cols(grid, n):
    return transpose(duplicate_rows(transpose(grid), n))

def get_row(grid, i):
    return grid[i][:]

def get_col(grid, j):
    return [grid[i][j] for i in range(len(grid))]

# ============================================================================
# 15. PATTERN / OBJECT UTILS
# ============================================================================
def objects_by_color(objects):
    result = defaultdict(list)
    for obj in objects:
        result[obj.get("color", -1)].append(obj)
    return dict(result)

def object_signature(obj, bg=0):
    g = obj["grid"]
    return grid_to_tuple(crop_to_content(g, bg))

# ============================================================================
# 16. ALL STRATEGY FUNCTIONS
# ============================================================================

def try_output_constant(train):
    if len(train) < 2:
        return None
    out0 = train[0]["output"]
    if all(grids_equal(p["output"], out0) for p in train[1:]):
        return lambda inp, o=copy_grid(out0): copy_grid(o)
    return None

def try_rigid_transforms(train):
    for name, op in ALL_RIGID_TRANSFORMS:
        if all(grids_equal(op(p["input"]), p["output"]) for p in train):
            return op
    return None

def try_color_map_only(train):
    cmap = infer_color_map(train)
    if all(cmap.get(k, k) == k for k in cmap):
        return None
    if all(grids_equal(color_map(p["input"], cmap), p["output"]) for p in train):
        return lambda inp, m=cmap: color_map(inp, m)
    return None

def try_color_swap(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    inp_colors = unique_colors(train[0]["input"])
    for c1 in inp_colors:
        for c2 in inp_colors:
            if c1 >= c2:
                continue
            mapping = {c1: c2, c2: c1}
            if all(grids_equal(color_map(p["input"], mapping), p["output"]) for p in train):
                return lambda inp, m=dict(mapping): color_map(inp, m)
    return None

def try_replace_single_color_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    diffs = defaultdict(set)
    for p in train:
        inp, out = p["input"], p["output"]
        h, w = grid_dims(inp)
        for i in range(h):
            for j in range(w):
                if inp[i][j] != out[i][j]:
                    diffs[inp[i][j]].add(out[i][j])
    if len(diffs) == 1:
        old_c = list(diffs.keys())[0]
        targets = diffs[old_c]
        if len(targets) == 1:
            new_c = list(targets)[0]
            if all(grids_equal(replace_color(p["input"], old_c, new_c), p["output"]) for p in train):
                return lambda inp, o=old_c, n=new_c: replace_color(inp, o, n)
    return None

def try_transform_then_color_map(train):
    for name, op in ALL_RIGID_TRANSFORMS[1:]:
        votes = defaultdict(list)
        ok = True
        for p in train:
            t = op(p["input"])
            out = p["output"]
            if grid_dims(t) != grid_dims(out):
                ok = False
                break
            h, w = grid_dims(t)
            for i in range(h):
                for j in range(w):
                    votes[t[i][j]].append(out[i][j])
        if not ok:
            continue
        cmap = {k: Counter(v).most_common(1)[0][0] for k, v in votes.items()}
        if all(grids_equal(color_map(op(p["input"]), cmap), p["output"]) for p in train):
            return lambda inp, o=op, m=cmap: color_map(o(inp), m)
    return None

def try_crop_to_content_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        if not grids_equal(crop_to_content(p["input"], bg), p["output"]):
            return None
    return lambda inp: crop_to_content(inp, background_color(inp))

def try_crop_to_specific_color(train):
    all_colors = set()
    for p in train:
        all_colors |= unique_colors(p["input"])
    for c in all_colors:
        if all(grids_equal(crop_color(p["input"], c), p["output"]) for p in train):
            return lambda inp, col=c: crop_color(inp, col)
    return None

def try_crop_then_transform(train):
    for tname, tfn in ALL_RIGID_TRANSFORMS:
        ok = True
        for p in train:
            bg = background_color(p["input"])
            cropped = crop_to_content(p["input"], bg)
            if not grids_equal(tfn(cropped), p["output"]):
                ok = False
                break
        if ok:
            def make_solver(tf):
                def solver(inp):
                    b = background_color(inp)
                    return tf(crop_to_content(inp, b))
                return solver
            return make_solver(tfn)
    return None

def try_scale(train):
    for fh in range(1, 9):
        for fw in range(1, 9):
            if fh == 1 and fw == 1:
                continue
            try:
                if all(
                    len(p["output"]) == len(p["input"]) * fh and
                    len(p["output"][0]) == len(p["input"][0]) * fw and
                    grids_equal(scale_grid(p["input"], fh, fw), p["output"])
                    for p in train
                ):
                    return lambda inp, a=fh, b=fw: scale_grid(inp, a, b)
            except Exception:
                continue
    return None

def try_tile(train):
    for rh in range(1, 9):
        for rw in range(1, 9):
            if rh == 1 and rw == 1:
                continue
            try:
                if all(
                    len(p["output"]) == len(p["input"]) * rh and
                    len(p["output"][0]) == len(p["input"][0]) * rw and
                    grids_equal(tile_grid(p["input"], rh, rw), p["output"])
                    for p in train
                ):
                    return lambda inp, a=rh, b=rw: tile_grid(inp, a, b)
            except Exception:
                continue
    return None

def try_downscale(train):
    for fh in range(2, 9):
        for fw in range(2, 9):
            ok = True
            for p in train:
                ih, iw = grid_dims(p["input"])
                oh, ow = grid_dims(p["output"])
                if ih % fh != 0 or iw % fw != 0 or ih // fh != oh or iw // fw != ow:
                    ok = False
                    break
                if not grids_equal(downscale_grid(p["input"], fh, fw), p["output"]):
                    ok = False
                    break
            if ok:
                return lambda inp, a=fh, b=fw: downscale_grid(inp, a, b)
    return None

def try_scale_then_color_map(train):
    for fh in range(2, 6):
        for fw in range(2, 6):
            ok = True
            for p in train:
                if (len(p["output"]) != len(p["input"]) * fh or
                        len(p["output"][0]) != len(p["input"][0]) * fw):
                    ok = False
                    break
            if not ok:
                continue
            scaled_train = [{"input": scale_grid(p["input"], fh, fw),
                              "output": p["output"]} for p in train]
            cmap = infer_color_map(scaled_train)
            if all(grids_equal(color_map(scale_grid(p["input"], fh, fw), cmap),
                               p["output"]) for p in train):
                def make_fn(a, b, m):
                    return lambda inp: color_map(scale_grid(inp, a, b), m)
                return make_fn(fh, fw, cmap)
    return None

def try_tile_then_color_map(train):
    for rh in range(1, 6):
        for rw in range(1, 6):
            if rh == 1 and rw == 1:
                continue
            ok = True
            for p in train:
                if (len(p["output"]) != len(p["input"]) * rh or
                        len(p["output"][0]) != len(p["input"][0]) * rw):
                    ok = False
                    break
            if not ok:
                continue
            tiled_train = [{"input": tile_grid(p["input"], rh, rw),
                             "output": p["output"]} for p in train]
            cmap = infer_color_map(tiled_train)
            if all(grids_equal(color_map(tile_grid(p["input"], rh, rw), cmap),
                               p["output"]) for p in train):
                def make_fn(a, b, m):
                    return lambda inp: color_map(tile_grid(inp, a, b), m)
                return make_fn(rh, rw, cmap)
    return None

def try_gravity_strategy(train):
    for func in [gravity_down, gravity_up, gravity_left, gravity_right]:
        if all(grids_equal(func(p["input"], background_color(p["input"])), p["output"])
               for p in train):
            return lambda inp, f=func: f(inp, background_color(inp))
    return None

def try_fill_enclosed_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    if all(grids_equal(fill_enclosed_regions(p["input"], background_color(p["input"])),
                       p["output"]) for p in train):
        return lambda inp: fill_enclosed_regions(inp, background_color(inp))
    return None

def try_remove_border_strategy(train):
    for p in train:
        ih, iw = grid_dims(p["input"])
        oh, ow = grid_dims(p["output"])
        if oh != ih - 2 or ow != iw - 2:
            return None
    if all(grids_equal(remove_border(p["input"]), p["output"]) for p in train):
        return remove_border
    return None

def try_add_border_strategy(train):
    for p in train:
        ih, iw = grid_dims(p["input"])
        oh, ow = grid_dims(p["output"])
        if oh != ih + 2 or ow != iw + 2:
            return None
    bc = Counter(extract_border(train[0]["output"])).most_common(1)[0][0]
    if all(grids_equal(add_border(p["input"], bc), p["output"]) for p in train):
        return lambda inp, c=bc: add_border(inp, c)
    return None

def try_multi_border_removal(train):
    for n_layers in range(2, 5):
        ok = True
        for p in train:
            g = p["input"]
            for _ in range(n_layers):
                g = remove_border(g)
            if not grids_equal(g, p["output"]):
                ok = False
                break
        if ok:
            def make_fn(n):
                def solver(inp):
                    g = inp
                    for _ in range(n):
                        g = remove_border(g)
                    return g
                return solver
            return make_fn(n_layers)
    return None

def try_symmetry_completion_strategy(train):
    for func in [make_4way_symmetric, make_horizontally_symmetric,
                 make_vertically_symmetric, complete_symmetry_bg_aware]:
        if all(grids_equal(func(p["input"]), p["output"]) for p in train):
            return func
    return None

def try_largest_object_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        objects = extract_multicolor_objects(p["input"], bg)
        if not objects:
            return None
        largest = max(objects, key=lambda o: o["size"])
        if not grids_equal(largest["grid"], p["output"]):
            return None
    def get_largest(inp):
        bg = background_color(inp)
        objects = extract_multicolor_objects(inp, bg)
        if not objects:
            return copy_grid(inp)
        return max(objects, key=lambda o: o["size"])["grid"]
    return get_largest

def try_smallest_object_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        objects = extract_multicolor_objects(p["input"], bg)
        if not objects:
            return None
        smallest = min(objects, key=lambda o: o["size"])
        if not grids_equal(smallest["grid"], p["output"]):
            return None
    def get_smallest(inp):
        bg = background_color(inp)
        objects = extract_multicolor_objects(inp, bg)
        if not objects:
            return copy_grid(inp)
        return min(objects, key=lambda o: o["size"])["grid"]
    return get_smallest

def try_nth_largest_object(train):
    for n in range(1, 5):
        ok = True
        for p in train:
            bg = background_color(p["input"])
            objects = sorted(extract_multicolor_objects(p["input"], bg),
                             key=lambda o: o["size"], reverse=True)
            if len(objects) < n:
                ok = False
                break
            if not grids_equal(objects[n - 1]["grid"], p["output"]):
                ok = False
                break
        if ok:
            def make_fn(nn):
                def solver(inp):
                    bg = background_color(inp)
                    objs = sorted(extract_multicolor_objects(inp, bg),
                                  key=lambda o: o["size"], reverse=True)
                    if len(objs) < nn:
                        return copy_grid(inp)
                    return objs[nn - 1]["grid"]
                return solver
            return make_fn(n)
    return None

def try_unique_object_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        objects = extract_multicolor_objects(p["input"], bg)
        sigs = [object_signature(o, bg) for o in objects]
        sig_count = Counter(sigs)
        unique_objs = [objects[i] for i, s in enumerate(sigs) if sig_count[s] == 1]
        if len(unique_objs) != 1:
            return None
        if not grids_equal(unique_objs[0]["grid"], p["output"]):
            return None
    def get_unique(inp):
        bg = background_color(inp)
        objects = extract_multicolor_objects(inp, bg)
        sigs = [object_signature(o, bg) for o in objects]
        sig_count = Counter(sigs)
        unique_objs = [objects[i] for i, s in enumerate(sigs) if sig_count[s] == 1]
        if len(unique_objs) == 1:
            return unique_objs[0]["grid"]
        return copy_grid(inp)
    return get_unique

def try_most_common_object(train):
    for p in train:
        bg = background_color(p["input"])
        objects = extract_multicolor_objects(p["input"], bg)
        if not objects:
            return None
        sigs = [object_signature(o, bg) for o in objects]
        sig_count = Counter(sigs)
        most_common_sig = sig_count.most_common(1)[0][0]
        candidates = [objects[i] for i, s in enumerate(sigs) if s == most_common_sig]
        if not any(grids_equal(c["grid"], p["output"]) for c in candidates):
            return None
    def get_most_common(inp):
        bg = background_color(inp)
        objects = extract_multicolor_objects(inp, bg)
        if not objects:
            return copy_grid(inp)
        sigs = [object_signature(o, bg) for o in objects]
        sig_count = Counter(sigs)
        most_common_sig = sig_count.most_common(1)[0][0]
        for i, s in enumerate(sigs):
            if s == most_common_sig:
                return objects[i]["grid"]
        return copy_grid(inp)
    return get_most_common

def try_denoise_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    for threshold in [1, 2, 3, 4]:
        def make_denoise(t):
            def denoise(inp):
                bg = background_color(inp)
                cc = color_counts(inp)
                h, w = grid_dims(inp)
                return [[bg if cc[inp[i][j]] <= t and inp[i][j] != bg else inp[i][j]
                          for j in range(w)] for i in range(h)]
            return denoise
        fn = make_denoise(threshold)
        if all(grids_equal(fn(p["input"]), p["output"]) for p in train):
            return fn
    return None

def try_mirror_and_extend_strategy(train):
    combos = [
        ("h_right",      lambda x: concat_horizontal([x, flip_h(x)])),
        ("h_left",       lambda x: concat_horizontal([flip_h(x), x])),
        ("v_down",       lambda x: concat_vertical([x, flip_v(x)])),
        ("v_up",         lambda x: concat_vertical([flip_v(x), x])),
        ("4way",         lambda x: concat_vertical([
            concat_horizontal([x, flip_h(x)]),
            concat_horizontal([flip_v(x), rotate180(x)])
        ])),
        ("4way_alt",     lambda x: concat_vertical([
            concat_horizontal([flip_h(x), x]),
            concat_horizontal([rotate180(x), flip_v(x)])
        ])),
        ("4way_v2",      lambda x: concat_vertical([
            concat_horizontal([x, flip_h(x)]),
            concat_horizontal([flip_v(x), flip_h(flip_v(x))])
        ])),
        ("tile_2x2_rot", lambda x: concat_vertical([
            concat_horizontal([x, rotate90(x)]),
            concat_horizontal([rotate270(x), rotate180(x)])
        ])),
    ]
    for name, func in combos:
        if all(grids_equal(func(p["input"]), p["output"]) for p in train):
            return func
    return None

def try_overlay_split_halves_strategy(train):
    split_fns = [
        ("h2", lambda g: split_grid_horizontal(g, 2)),
        ("v2", lambda g: split_grid_vertical(g, 2)),
        ("h3", lambda g: split_grid_horizontal(g, 3)),
        ("v3", lambda g: split_grid_vertical(g, 3)),
        ("h4", lambda g: split_grid_horizontal(g, 4)),
        ("v4", lambda g: split_grid_vertical(g, 4)),
    ]
    merge_fns = [or_grids, and_grids, xor_grids, subtract_grids]
    for sf_name, split_fn in split_fns:
        for merge_fn in merge_fns:
            try:
                ok = True
                for p in train:
                    parts = split_fn(p["input"])
                    if len(parts) < 2:
                        ok = False
                        break
                    ph, pw = grid_dims(parts[0])
                    if not all(grid_dims(pt) == (ph, pw) for pt in parts):
                        ok = False
                        break
                    result = parts[0]
                    for pt in parts[1:]:
                        result = merge_fn(result, pt, 0)
                    if not grids_equal(result, p["output"]):
                        ok = False
                        break
                if ok:
                    def make_solver(sf, mf):
                        def solver(inp):
                            ps = sf(inp)
                            r = ps[0]
                            for pt in ps[1:]:
                                r = mf(r, pt, 0)
                            return r
                        return solver
                    return make_solver(split_fn, merge_fn)
            except Exception:
                continue
    return None

def try_extract_repeating_unit_strategy(train):
    for p in train:
        unit = find_repeating_unit(p["input"])
        if not grids_equal(unit, p["output"]):
            return None
    return lambda inp: find_repeating_unit(inp)

def try_row_col_duplication_strategy(train):
    for n in range(2, 7):
        if all(grids_equal(duplicate_rows(p["input"], n), p["output"]) for p in train):
            return lambda inp, nn=n: duplicate_rows(inp, nn)
        if all(grids_equal(duplicate_cols(p["input"], n), p["output"]) for p in train):
            return lambda inp, nn=n: duplicate_cols(inp, nn)
    return None

def try_per_object_transform_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    for tname, tfn in ALL_RIGID_TRANSFORMS[1:]:
        ok = True
        for p in train:
            bg = background_color(p["input"])
            h, w = grid_dims(p["input"])
            objects = extract_multicolor_objects(p["input"], bg)
            result = [[bg] * w for _ in range(h)]
            for obj in objects:
                tobj = tfn(obj["grid"])
                th, tw = grid_dims(tobj)
                r1, c1, _, _ = obj["bbox"]
                for di in range(th):
                    for dj in range(tw):
                        ni, nj = r1 + di, c1 + dj
                        if 0 <= ni < h and 0 <= nj < w and tobj[di][dj] != bg:
                            result[ni][nj] = tobj[di][dj]
            if not grids_equal(result, p["output"]):
                ok = False
                break
        if ok:
            def make_solver(tf):
                def solver(inp):
                    bg = background_color(inp)
                    h, w = grid_dims(inp)
                    objs = extract_multicolor_objects(inp, bg)
                    res = [[bg] * w for _ in range(h)]
                    for obj in objs:
                        to = tf(obj["grid"])
                        th, tw = grid_dims(to)
                        r1, c1, _, _ = obj["bbox"]
                        for di in range(th):
                            for dj in range(tw):
                                ni, nj = r1 + di, c1 + dj
                                if 0 <= ni < h and 0 <= nj < w and to[di][dj] != bg:
                                    res[ni][nj] = to[di][dj]
                    return res
                return solver
            return make_solver(tfn)
    return None

def try_pixel_checker_rule(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    rules = {}
    ok = True
    for p in train:
        inp, out = p["input"], p["output"]
        h, w = grid_dims(inp)
        for i in range(h):
            for j in range(w):
                key = (inp[i][j], (i + j) % 2)
                if key in rules:
                    if rules[key] != out[i][j]:
                        ok = False
                        break
                else:
                    rules[key] = out[i][j]
            if not ok:
                break
        if not ok:
            break
    if ok and rules:
        def apply_rule(inp, r=dict(rules)):
            h, w = grid_dims(inp)
            return [[r.get((inp[i][j], (i + j) % 2), inp[i][j]) for j in range(w)]
                    for i in range(h)]
        if all(grids_equal(apply_rule(p["input"]), p["output"]) for p in train):
            return apply_rule
    return None

def try_position_color_rule(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    for mod in [2, 3, 4]:
        rules = {}
        ok = True
        for p in train:
            inp, out = p["input"], p["output"]
            h, w = grid_dims(inp)
            for i in range(h):
                for j in range(w):
                    key = (inp[i][j], i % mod, j % mod)
                    if key in rules:
                        if rules[key] != out[i][j]:
                            ok = False
                            break
                    else:
                        rules[key] = out[i][j]
                if not ok:
                    break
            if not ok:
                break
        if ok and rules:
            def make_fn(r, m):
                def apply(inp):
                    h, w = grid_dims(inp)
                    return [[r.get((inp[i][j], i % m, j % m), inp[i][j]) for j in range(w)]
                            for i in range(h)]
                return apply
            fn = make_fn(dict(rules), mod)
            if all(grids_equal(fn(p["input"]), p["output"]) for p in train):
                return fn
    return None

def try_split_and_overlay_dividers(train):
    for p in train:
        h_div, v_div = detect_grid_divisions(p["input"])
        if not h_div and not v_div:
            return None

    merge_fns = [or_grids, and_grids, xor_grids, subtract_grids]

    for merge_fn in merge_fns:
        ok = True

        for p in train:
            h_div, v_div = detect_grid_divisions(p["input"])
            cells = split_by_dividers(p["input"], h_div, v_div)
            flat = [cell for row in cells for cell in row]

            if len(flat) < 2:
                ok = False
                break

            ch, cw = grid_dims(flat[0])
            if not all(grid_dims(cell) == (ch, cw) for cell in flat):
                ok = False
                break

            result = flat[0]
            for cell in flat[1:]:
                result = merge_fn(result, cell, 0)

            if not grids_equal(result, p["output"]):
                ok = False
                break

        if ok:
            def make_solver(mf):
                def solver(inp):
                    h_div, v_div = detect_grid_divisions(inp)
                    cells = split_by_dividers(inp, h_div, v_div)
                    flat = [cell for row in cells for cell in row]

                    if not flat:
                        return copy_grid(inp)

                    result = flat[0]
                    for cell in flat[1:]:
                        result = mf(result, cell, 0)

                    return result
                return solver

            return make_solver(merge_fn)

    return None

def try_subgrid_extraction(train):
    for p in train:
        inp, out = p["input"], p["output"]
        oh, ow = grid_dims(out)
        ih, iw = grid_dims(inp)
        found = False
        for r in range(ih - oh + 1):
            for c in range(iw - ow + 1):
                if grids_equal(crop(inp, r, c, r + oh, c + ow), out):
                    found = True
                    break
            if found:
                break
        if not found:
            return None
    out_dims = set(grid_dims(p["output"]) for p in train)
    if len(out_dims) != 1:
        return None
    ooh, oow = list(out_dims)[0]
    for anchor in ["top_left", "top_right", "bottom_left", "bottom_right", "center"]:
        ok = True
        for p in train:
            ih, iw = grid_dims(p["input"])
            if anchor == "top_left":
                r, c = 0, 0
            elif anchor == "top_right":
                r, c = 0, iw - oow
            elif anchor == "bottom_left":
                r, c = ih - ooh, 0
            elif anchor == "bottom_right":
                r, c = ih - ooh, iw - oow
            else:
                r, c = (ih - ooh) // 2, (iw - oow) // 2
            if r < 0 or c < 0:
                ok = False
                break
            if not grids_equal(crop(p["input"], r, c, r + ooh, c + oow), p["output"]):
                ok = False
                break
        if ok:
            def make_crop_solver(anc, dh, dw):
                def solver(inp):
                    ih, iw = grid_dims(inp)
                    if anc == "top_left":
                        return crop(inp, 0, 0, dh, dw)
                    elif anc == "top_right":
                        return crop(inp, 0, iw - dw, dh, iw)
                    elif anc == "bottom_left":
                        return crop(inp, ih - dh, 0, ih, dw)
                    elif anc == "bottom_right":
                        return crop(inp, ih - dh, iw - dw, ih, iw)
                    else:
                        return crop(inp, (ih - dh) // 2, (iw - dw) // 2,
                                    (ih - dh) // 2 + dh, (iw - dw) // 2 + dw)
                return solver
            return make_crop_solver(anchor, ooh, oow)
    return None

def try_recolor_by_size(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    size_to_color = {}
    for p in train:
        bg = background_color(p["input"])
        objs_in = extract_objects(p["input"], bg)
        objs_out = extract_objects(p["output"], bg)
        for oi in objs_in:
            for oo in objs_out:
                if oi["bbox"] == oo["bbox"] and oi["size"] == oo["size"]:
                    s = oi["size"]
                    if s in size_to_color and size_to_color[s] != oo["color"]:
                        return None
                    size_to_color[s] = oo["color"]
                    break
    if not size_to_color:
        return None
    def recolor(inp, s2c=dict(size_to_color)):
        bg = background_color(inp)
        result = copy_grid(inp)
        objs = extract_objects(inp, bg)
        for obj in objs:
            if obj["size"] in s2c:
                for r, c in obj["positions"]:
                    result[r][c] = s2c[obj["size"]]
        return result
    if all(grids_equal(recolor(p["input"]), p["output"]) for p in train):
        return recolor
    return None

def try_output_fixed_size_crop(train):
    out_dims = set(grid_dims(p["output"]) for p in train)
    if len(out_dims) != 1:
        return None
    oh, ow = list(out_dims)[0]
    for p in train:
        bg = background_color(p["input"])
        all_c = unique_colors(p["input"]) - {bg}
        for color in all_c:
            positions = color_positions(p["input"], color)
            if not positions:
                continue
            cr = sum(r for r, c in positions) // len(positions)
            cc_avg = sum(c for r, c in positions) // len(positions)
            ih, iw = grid_dims(p["input"])
            r1 = max(0, min(cr - oh // 2, ih - oh))
            c1 = max(0, min(cc_avg - ow // 2, iw - ow))
            if r1 + oh <= ih and c1 + ow <= iw:
                if grids_equal(crop(p["input"], r1, c1, r1 + oh, c1 + ow), p["output"]):
                    def make_solver(col, ooh, oow):
                        def solver(inp):
                            pos = color_positions(inp, col)
                            if not pos:
                                return crop(inp, 0, 0, ooh, oow)
                            cr2 = sum(r for r, c in pos) // len(pos)
                            cc2 = sum(c for r, c in pos) // len(pos)
                            ih2, iw2 = grid_dims(inp)
                            r12 = max(0, min(cr2 - ooh // 2, ih2 - ooh))
                            c12 = max(0, min(cc2 - oow // 2, iw2 - oow))
                            return crop(inp, r12, c12, r12 + ooh, c12 + oow)
                        return solver
                    solver = make_solver(color, oh, ow)
                    if all(grids_equal(solver(pp["input"]), pp["output"]) for pp in train):
                        return solver
    return None

def try_fill_by_neighbor_rule(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    for p in train:
        bg = background_color(p["input"])
        inp, out = p["input"], p["output"]
        h, w = grid_dims(inp)
        for i in range(h):
            for j in range(w):
                if inp[i][j] != bg and inp[i][j] != out[i][j]:
                    return None

    def fill_nearest(inp):
        bg = background_color(inp)
        h, w = grid_dims(inp)
        result = copy_grid(inp)
        non_bg = [(i, j, inp[i][j]) for i in range(h) for j in range(w) if inp[i][j] != bg]
        if not non_bg:
            return result
        for i in range(h):
            for j in range(w):
                if inp[i][j] == bg:
                    best = min(non_bg, key=lambda x: abs(x[0] - i) + abs(x[1] - j))
                    result[i][j] = best[2]
        return result
    if all(grids_equal(fill_nearest(p["input"]), p["output"]) for p in train):
        return fill_nearest
    return None
# ============================================================================
# 17. MASTER SOLVER
# ============================================================================

STRATEGIES = [
    try_output_constant,
    try_rigid_transforms,
    try_color_map_only,
    try_color_swap,
    try_replace_single_color_strategy,
    try_transform_then_color_map,
    try_crop_to_content_strategy,
    try_crop_to_specific_color,
    try_crop_then_transform,
    try_scale,
    try_tile,
    try_downscale,
    try_scale_then_color_map,
    try_tile_then_color_map,
    try_gravity_strategy,
    try_fill_enclosed_strategy,
    try_remove_border_strategy,
    try_add_border_strategy,
    try_multi_border_removal,
    try_symmetry_completion_strategy,
    try_largest_object_strategy,
    try_smallest_object_strategy,
    try_nth_largest_object,
    try_unique_object_strategy,
    try_most_common_object,
    try_denoise_strategy,
    try_mirror_and_extend_strategy,
    try_overlay_split_halves_strategy,
    try_extract_repeating_unit_strategy,
    try_row_col_duplication_strategy,
    try_per_object_transform_strategy,
    try_pixel_checker_rule,
    try_position_color_rule,
]

def safe_fallback_prediction(inp):
    h, w = len(inp), len(inp[0])
    bg = most_common_color(inp)

    pred1 = copy_grid(inp)
    pred2 = [[bg] * w for _ in range(h)]

    return pred1, pred2

def find_program(task):
    train = task["train"]

    for strategy in STRATEGIES:
        try:
            program = strategy(train)
            if program is not None:
                return program
        except Exception:
            continue

    return None

def solve_task(task):
    results = []
    program = find_program(task)

    for test_case in task["test"]:
        inp = test_case["input"]

        if program is not None:
            try:
                pred1 = program(inp)
                pred2 = pred1
            except Exception:
                pred1, pred2 = safe_fallback_prediction(inp)
        else:
            pred1, pred2 = safe_fallback_prediction(inp)

        results.append({
            "attempt_1": pred1,
            "attempt_2": pred2
        })

    return results

In [2]:
# ============================================================================
# 18. BUILD SUBMISSION
# ============================================================================

submission = {}

for task_id, task in data.items():
    submission[task_id] = solve_task(task)

with open("submission.json", "w") as f:
    json.dump(submission, f)

print("submission.json created successfully!")

submission.json created successfully!
